# Indicadores Macroeconômicos
**Cálculos Econômicos Estruturais**

Este módulo implementa o cálculo de indicadores macroeconômicos derivados a partir das séries temporais de inflação (IPCA) e taxa básica de juros (SELIC).

Objetivo:  
Construir métricas econômicas que permitam avaliar a relação entre inflação, política monetária e estabilidade macroeconômica ao longo do tempo.

Período analisado:
- 2016 – 2030
- Dados históricos
- Projeções macroeconômicas
- Diferentes cenários econômicos,
  
Estrutura da Base de Dados:
- Periodicidade: mensal
- Formato: série temporal
- Cobertura: dados históricos e projeções

Variáveis principais:
- SELIC — taxa básica de juros da economia
- IPCA — índice oficial de inflação
- Cenário (Otimista, Base, Pessimista) — classificação do cenário macroeconômico
- Taxa projetada por cenário

**INDICADORES CALCULADOS**

**Juros Real**

Mede o retorno da taxa de juros descontando o efeito da inflação.

**Volatilidade do Juros Real**

Calcula a variabilidade da série de juros reais ao longo do tempo.

**IPCA Acumulado em 12 Meses**

Representa a inflação anualizada a partir da soma ou composição dos últimos 12 meses.

**SELIC Acumulado em 12 Meses**

Representa a inflação anualizada a partir da soma ou composição dos últimos 12 meses.

**Aplicações de calculos Estatísticos e Tendências:**

O módulo também aplica técnicas estatísticas para análise das séries econômicas:
- Tendência linear mensal e anual
- Correlação rolante
- Z-score
- IQR

**Output:**

O resultado do processo é um arquivo estruturado contendo os indicadores calculados ao longo do tempo, segmentados por cenário macroeconômico.

**O output inclui:**
- Séries temporais processadas
- Indicadores derivados
- Métricas estatísticas
- Consolidações por cenário


## Pivot necessario para os Cálculos

In [40]:
# Transformação do dataframe do formato longo (long format)
# para formato largo (wide format), permitindo que cada indicador
# econômico (ex: SELIC e IPCA) fique em uma coluna separada.

df_pivot_col_final_selic_ipca = df_pivot_col_final.pivot_table(

    # Define as colunas que formarão o índice da nova tabela.
    # Cada combinação única de data, período do cenário e cenário
    # representará uma linha no dataframe resultante.
    index=['data','periodo_cenario','cenarios'],

    # Define qual coluna do dataframe original será usada para criar
    # novas colunas no dataframe resultante. Nesse caso, cada tipo
    # de indicador (ex: SELIC, IPCA) vira uma coluna.
    columns='indicador',

    # Define qual coluna contém os valores numéricos que preencherão
    # as células da nova tabela (valores das taxas).
    values='vl_taxa'

# Após a criação da tabela dinâmica (pivot), o índice é resetado
# para transformar as colunas de índice novamente em colunas normais
# do dataframe, facilitando análises e cálculos posteriores.
).reset_index()

## Preencher os valores nulos

Alguns registros da base apresentam valores nulos para os indicadores IPCA e SELIC.

Isso ocorre porque, em determinadas datas, um dos indicadores pode ainda não possuir projeção disponível ou já estar registrado como valor observado, não havendo uma projeção correspondente.

Para garantir a continuidade da série temporal e permitir o cálculo de indicadores derivados (como juros reais e volatilidade), foi adotada a estratégia de preenchimento utilizando o último valor disponível da série.

O procedimento consiste em preencher os valores nulos utilizando o valor imediatamente anterior, respeitando o cenário econômico (base, otimista ou pessimista). Dessa forma, preserva-se a coerência temporal e evita-se a mistura de informações entre cenários distintos.

Como os dados são previamente ordenados pela coluna de data, o preenchimento sequencial mantém a consistência cronológica da série.

In [50]:
# Primeiro garantir que a série está ordenada corretamente.
df_preench_vl_nul_ipca_selic = df_pivot_col_final_selic_ipca.sort_values(
    ['cenarios','data']
)

In [51]:
# Preencher usando o valor anterior (forward fill) respeitando o cenario
df_preench_vl_nul_ipca_selic[['IPCA','SELIC']] = (
    df_preench_vl_nul_ipca_selic
    .groupby('cenarios')[['IPCA','SELIC']]
    .ffill()
)

In [67]:
# Conversão da taxa de inflação (IPCA) de percentual para formato decimal. SELIC não precisa dessa correção pois ja esta no formato correto.
# Os valores da base estão representados em percentual (ex: 0.33 = 0,33%),
# porém, para realizar cálculos financeiros como o juros real, é necessário
# utilizar a taxa em formato decimal. Portanto, divide-se o valor por 100.
# Exemplo: 0.33% → 0.0033
df_preench_vl_nul_ipca_selic['IPCA'] = df_preench_vl_nul_ipca_selic['IPCA'] / 100

## $$Cálculos Econômicos$$

### Juros Real
**Métrica central de postura monetária**

* O juros real mede o poder de ganho da taxa de juros após descontar o efeito da inflação ou seja, ele indica quanto o dinheiro realmente cresce em termos de poder de compra.
* No caso da SELIC e do IPCA, o juros real mede quanto a taxa básica de juros da economia supera (ou não) a inflação. Mostra o ganho ou custo real do dinheiro na economia brasileira.

$$
Juro\ Real = (\frac{1 + SELIC}{1 + IPCA} - 1)
$$



In [89]:
# Definindo o nome do DataFrame para este calculo
df_calc_juros_real = df_preench_vl_nul_ipca_selic

In [90]:
# Cálculo da taxa de juros real

# A taxa de juros real representa o retorno da taxa de juros nominal
# (SELIC) descontando o efeito da inflação (IPCA).
# Esse indicador é amplamente utilizado para avaliar o custo real do
# dinheiro na economia e a postura da política monetária.

# A fórmula utilizada é:
# Juros Real = (1 + taxa nominal) / (1 + inflação) - 1

df_calc_juros_real['juros_real'] = (
    (1 + df_calc_juros_real['SELIC']) /  # soma 1 à taxa nominal (SELIC)
    (1 + df_calc_juros_real['IPCA']) -   # soma 1 à taxa de inflação (IPCA)
    1                                    # remove o fator base para obter a taxa real
)

# O resultado é armazenado na coluna 'juros_real', representando
# a taxa de juros ajustada pela inflação para cada data e cenário.

### Volatilidade do Juros Real

*Mede a instabilidade da postura monetária ao longo do tempo.*

A volatilidade é calculada como o **desvio padrão** do juro real dentro de um grupo (por exemplo, por cenário, por ano ou em janela móvel):

- **Por cenário:**
$$
\text{Vol}_{r,\text{cenário}} = \sigma(r) \quad \text{para cada cenário}
$$

- **Por ano e cenário:**
$$
\text{Vol}_{r,\text{ano,cenário}} = \sigma(r) \quad \text{para cada ano dentro de cada cenário}
$$

- **Volatilidade rolante (móvel):**
$$
\text{Vol}_{r,\text{janela}} = \sigma(r_{\text{últimos N períodos}})
$$

Onde:  
- $\( r \)$ = Juros Real = SELIC − Inflação Esperada (IPCA)  
- $\( \sigma \)$ = desvio padrão  

Essa métrica ajuda a entender quão **instável** está a política monetária em diferentes cenários ou períodos.

A volatilidade do juros real foi calculada utilizando o desvio padrão da série. Foram consideradas três abordagens: (i) volatilidade agregada por cenário, permitindo avaliar a instabilidade das condições monetárias em cada cenário econômico; (ii) volatilidade anual por cenário, permitindo identificar períodos de maior variabilidade; e (iii) volatilidade rolante de 12 meses, utilizada para capturar mudanças dinâmicas na estabilidade do juros real ao longo do tempo.


#### Volatilidade do Juros Real Por cenário

In [116]:
# Calcula a volatilidade do juros real para cada cenário macroeconômico
# A volatilidade é medida pelo desvio padrão da série de juros reais

df_volatil_juros_real_cenario = (
    
    df_calc_juros_real
    
    # Agrupa os dados pelo tipo de cenário (ex: otimista, base, pessimista)
    .groupby('cenarios')['juros_real']
    
    # Calcula o desvio padrão da variável 'juros_real'
    # Este valor representa a volatilidade da taxa de juros real dentro de cada cenário
    .std()
    
    # Converte o resultado novamente para um dataframe
    # e define o nome da coluna resultante como 'vol_jrs_real_cnr'
    .reset_index(name='vol_jrs_real_cnr')
)


In [117]:
# Realiza um merge (join) entre o dataframe de cálculo do juros real
# e o dataframe que contém a volatilidade do juros real por cenário

df_volatil_juros_real_cenario = df_calc_juros_real.merge(
    
    # DataFrame que contém a volatilidade do juros real calculada por cenário
    df_volatil_juros_real_cenario,
    
    # Define o tipo de join como 'left'
    # Mantém todas as linhas do dataframe df_calc_juros_real
    # e adiciona a coluna de volatilidade quando houver correspondência
    how='left',       
    
    # Chave utilizada para realizar o relacionamento entre os dataframes
    # Neste caso, o join é feito pela coluna 'cenarios'
    on='cenarios'
)

#### Volatilidade do Juros Real por ano e cenário

In [122]:
# Cria uma nova coluna chamada 'ano' extraindo apenas o ano da coluna de data
# A função .dt.year é usada para acessar o componente de ano de uma coluna do tipo datetime

df_volatil_juros_real_cenario['ano'] = df_volatil_juros_real_cenario['data'].dt.year

In [123]:
# Calcula a volatilidade anual do juros real para cada cenário macroeconômico

df_volatil_juros_real_cenario_ano = (
    
    df_volatil_juros_real_cenario
    
    # Agrupa os dados pelo ano e pelo tipo de cenário
    # Isso permite calcular a volatilidade do juros real dentro de cada combinação ano-cenário
    .groupby(['ano','cenarios'])['juros_real']
    
    # Calcula o desvio padrão da série 'juros_real'
    # O desvio padrão é usado como medida de volatilidade
    .std()
    
    # Converte o resultado novamente em um DataFrame
    # e nomeia a coluna resultante como 'vol_juros_real'
    .reset_index(name='vol_jrs_real_cnr_ano')
)

In [124]:
# Realiza um merge entre o dataframe principal e o dataframe
# que contém a volatilidade anual do juros real por cenário

df_volatil_juros_real_cenario_ano = df_volatil_juros_real_cenario.merge(
    
    # DataFrame que possui a volatilidade anual calculada
    df_volatil_juros_real_cenario_ano,
    
    # Define o tipo de join como 'left'
    # Mantém todas as linhas do dataframe principal
    # e adiciona as informações de volatilidade anual quando houver correspondência
    how='left',
    
    # Define as chaves utilizadas para realizar o relacionamento entre os dataframes
    # O join é feito pelo cenário macroeconômico e pelo ano
    on=['cenarios', 'ano']
).drop('ano', axis=1)

#### Volatilidade do Juros Real rolante (móvel)

In [128]:
# Ordena o dataframe pelos campos 'cenarios' e 'data'
# Isso garante que as observações fiquem organizadas cronologicamente dentro de cada cenário
# A ordenação é importante principalmente antes de aplicar cálculos de séries temporais,
# como média móvel, volatilidade rolante ou outras métricas dependentes da ordem temporal

df_volatil_juros_real_movel = df_volatil_juros_real_cenario_ano.sort_values(['cenarios','data'])

In [129]:
# Calcula a volatilidade móvel de 12 meses do juros real para cada cenário macroeconômico
# A volatilidade é medida pelo desvio padrão em uma janela móvel de 12 períodos (meses)

df_volatil_juros_real_movel['vol_juros_real_12m'] = (
    
    df_volatil_juros_real_movel
    
    # Agrupa os dados por cenário (ex: otimista, base, pessimista)
    # Isso garante que o cálculo da volatilidade seja feito separadamente para cada cenário
    .groupby('cenarios')['juros_real']
    
    # Aplica uma janela móvel de 12 períodos
    # Ou seja, para cada linha será considerado o comportamento dos últimos 12 meses
    .rolling(12)
    
    # Calcula o desvio padrão dentro dessa janela
    # O desvio padrão representa a volatilidade do juros real no período
    .std()
    
    # Ajusta o índice para alinhar o resultado com o dataframe original
    # remove o nível adicional criado pelo groupby
    .reset_index(level=0, drop=True)
)

### IPCA Acumulado 12 Meses

*Pressão inflacionária anualizada.*

O acumulado 12 meses do IPCA mede quanto a inflação efetivamente cresceu nos últimos 12 meses, refletindo a **pressão inflacionária anualizada**.  
Se a taxa mensal de inflação for $\(\pi_t\)$, o cálculo composto correto é:

$$
IPCA_{12m} = \prod_{i=t-11}^{t} (1 + \pi_i) - 1
$$
$$ Exemplo: 
IPCA_{12m}^{fev} = (1 + 0.0127) \times (1 + 0.0090) - 1
$$

Onde:
- $\(\pi_i\)$ = IPCA mensal do período \(i\)  
- $\(t\)$ = mês corrente  
- $\(\prod\)$ = produto acumulado dos últimos 12 meses  

> Essa métrica mostra o nível de inflação acumulada nos últimos 12 meses, ajudando a avaliar o **impacto da inflação anualizada** no cenário econômico.



In [152]:
# Ordena o dataframe pelos campos 'cenarios' e 'data'
# Isso garante que as observações fiquem organizadas cronologicamente dentro de cada cenário
# A ordenação é importante para cálculos de séries temporais,
# como acumulados, médias móveis ou janelas rolantes (rolling)

df_acum_ipca_12m = df_volatil_juros_real_movel.sort_values(['cenarios', 'data'])

In [153]:
# Calcula o IPCA acumulado em 12 meses para cada cenário macroeconômico

df_acum_ipca_12m['IPCA_acum_12m'] = (
    
    # Agrupa os dados pelo tipo de cenário (ex: otimista, base, pessimista)
    df_acum_ipca_12m.groupby('cenarios')['IPCA']
    
    # Mantém o mesmo tamanho do dataframe original após o cálculo
    .transform(
        
        # Aplica a transformação na série de IPCA de cada cenário
        lambda x: (x + 1)
        
        # Cria uma janela móvel de 12 meses
        # min_periods=1 permite calcular o acumulado mesmo nos primeiros meses
        .rolling(12, min_periods=1)
        
        # Calcula o produto acumulado dos valores dentro da janela
        # equivalente a multiplicar (1 + inflação mensal)
        .apply(lambda y: y.prod(), raw=True)
        
        # Subtrai 1 para obter o valor final da inflação acumulada
        - 1
    )
)

### SELIC Acumulada 12 Meses

*Intensidade anualizada da política monetária.*

A SELIC acumulada em 12 meses mede o efeito composto da taxa básica de juros nos últimos 12 meses, refletindo a **intensidade anualizada da política monetária**.

Se a taxa mensal da SELIC for \( r_t \), o cálculo composto correto é:

$$
SELIC_{12m} = \prod_{i=t-11}^{t} (1 + r_i) - 1
$$

$$ Exemplo : 
SELIC_{12m}^{fev} = (1 + 0.052531) \times (1 + 0.052531) - 1
$$
Onde:
- $\( r_i \)$ = taxa SELIC mensal do período \( i \)  
- $\( t \)$ = mês corrente  
- $\( \prod \)$ = produto acumulado dos últimos 12 meses  

> Essa métrica mostra o nível efetivo de juros acumulado nos últimos 12 meses, permitindo avaliar o **grau de aperto ou estímulo monetário** vigente na economia.


In [157]:
# Ordena o dataframe pelos campos 'cenarios' e 'data'
# Isso garante que as observações fiquem organizadas cronologicamente dentro de cada cenário
# A ordenação é importante para cálculos de séries temporais,
# como acumulados, médias móveis ou janelas rolantes (rolling)
df_acum_selic_12m = df_acum_ipca_12m.sort_values(['cenarios', 'data'])

In [158]:
# Calcula a SELIC acumulada em 12 meses para cada cenário macroeconômico

df_acum_selic_12m['SELIC_acum_12m'] = (
    
    # Agrupa os dados pelo tipo de cenário (ex: otimista, base, pessimista)
    df_acum_selic_12m.groupby('cenarios')['SELIC']
    
    # Aplica a transformação para cada grupo de cenário
    .apply(
        lambda x: (x + 1) 
        
        # Cria uma janela móvel de 12 meses
        # min_periods=1 permite calcular o acumulado mesmo nos primeiros meses
        .rolling(12, min_periods=1)
        
        # Calcula o produto acumulado dos valores dentro da janela
        # equivalente à composição da taxa de juros ao longo do tempo
        .apply(lambda y: y.prod(), raw=True)
        
        # Subtrai 1 para obter a taxa acumulada final
        - 1
    )
    
    # Ajusta o índice para alinhar o resultado com o dataframe original
    .reset_index(level=0, drop=True)
)

## $$ Calculos Estatísticos e Tendências$$

### Tendência Mensal (Regressão Linear)
**Identificar se há tendência de alta ou queda ao longo do tempo.**
Para cada cenário:

$$Yₜ = β0 + β1t + εt$$

- **yt** → valor do indicador no mês 𝑡 (IPCA_acum_12m ou SELIC_12m)
- **t** → número do mês (1, 2, 3, …) dentro do cenário(nr_mes)
- **β0** → intercepto da regressão (valor de Y quando t = 0)
- **β1** → coeficiente angular, que representa a tendência mensal (quanto Y muda a cada mês)
  
Se:
- **β > 0** → tendência de alta  
- **β < 0** → tendência de queda  
- **β ≈ 0** → estabilidade
- **𝜀𝑡**  → erro residual
- 
Para interpretar em termos anuais, basta multiplicar **β₁** por 12 → tendência anualizada
Aplicável tanto para **IPCA** quanto para **SELIC** projetados

In [83]:
# Ordena os dados por cenário e data para garantir a sequência temporal correta
df_calc_tendencia_mensal = df_acum_selic_12m.sort_values(
    ['cenarios', 'data']
)

# Cria um índice sequencial de meses dentro de cada cenário
# Esse índice será usado como variável temporal para cálculos de tendência
df_calc_tendencia_mensal['nr_mes'] = (
    df_calc_tendencia_mensal
    .groupby('cenarios')
    .cumcount()
)

In [84]:
# Função para calcular tendência linear de uma série temporal
def calcular_tendencia(series):

    # Variável temporal (número sequencial de meses)
    X = df_calc_tendencia_mensal.loc[series.index, 'nr_mes'].values.reshape(-1, 1)

    # Valores da série
    y = series.values

    # Modelo de regressão linear
    modelo = LinearRegression()
    modelo.fit(X, y)

    # Retorna o coeficiente angular (inclinação da reta)
    # Representa a tendência média mensal da série
    return modelo.coef_[0]

In [87]:
# Cálculo da tendência linear das séries por cenário
# Tendência mensal do IPCA acumulado em 12 meses
df_calc_tendencia_mensal['tendencia_IPCA_12m'] = (
    df_calc_tendencia_mensal
    .groupby('cenarios')['IPCA_acum_12m']
    .transform(calcular_tendencia)
)

# Tendência mensal da SELIC acumulada em 12 meses
df_calc_tendencia_mensal['tendencia_SELIC_12m'] = (
    df_calc_tendencia_mensal
    .groupby('cenarios')['SELIC_acum_12m']
    .transform(calcular_tendencia)
)


In [88]:
# Anualização das tendências
# Converte a tendência mensal para tendência anual
df_calc_tendencia_mensal['tendencia_IPCA_12m_anual'] = (
    df_calc_tendencia_mensal['tendencia_IPCA_12m'] * 12
)

df_calc_tendencia_mensal['tendencia_SELIC_12m_anual'] = (
    df_calc_tendencia_mensal['tendencia_SELIC_12m'] * 12
)

# drop em coluna que não preciso mais
df_calc_tendencia_mensal = df_calc_tendencia_mensal.drop(columns=['nr_mes'], errors='ignore')

### Rolling Correlation (IPCA vs SELIC)
**Correlação móvel entre IPCA e SELIC**

A rolling correlation mede como a relação linear entre duas variáveis
**(IPCA acumulado em 12 meses e SELIC acumulada em 12 meses)** evolui ao longo do tempo, utilizando uma janela deslizante
de tamanho fixo.

Para cada período **t**, calcula-se a correlação de Pearson
entre as últimas n observações:

$$ρ_t = corr(X_{t-n+1}, ..., X_t ; Y_{t-n+1}, ..., Y_t)$$

**Onde:**

- X_t = variação do IPCA no período t  
- Y_t = variação da SELIC no período t  
- n = tamanho da janela  
- ρ_t = correlação estimada no período t  

**Interpretação:**

- ρ_t ≈ +1 → forte relação linear positiva  
- ρ_t ≈ -1 → forte relação linear negativa  
- ρ_t ≈ 0  → ausência de relação linear

**Observação:**

A rolling correlation permite identificar mudanças dinâmicas
na relação entre inflação e política monetária ao longo do tempo.

In [89]:
# Parâmetro da janela temporal
# Define a janela da correlação rolante (em meses)
window = 12  # correlação móvel de 12 meses

In [90]:
# DataFrame de trabalho para cálculo de correlação
# Cria uma cópia do dataframe original para evitar alterar
# o dataset principal durante os cálculos
df_corr = df_calc_tendencia_mensal.copy()

In [91]:
# Inicializar colunas de correlação rolante
df_corr['corr_IPCA_SELIC_acum_12m'] = float('nan')
df_corr['corr_IPCA_juros_real_12m'] = float('nan')

In [93]:
# Correlação rolante por cenário
for cenario, grupo in df_corr.groupby('cenarios'):

    # garante que os dados estão ordenados no tempo
    grupo = grupo.sort_values('data')

    # correlação rolante entre IPCA acumulado 12m e SELIC 12m
    corr_ipca_selic = (
        grupo['IPCA_acum_12m']
        .rolling(window)
        .corr(grupo['SELIC_acum_12m'])
    )

    # correlação rolante entre IPCA acumulado 12m e Juros Real
    corr_ipca_jr = (
        grupo['IPCA_acum_12m']
        .rolling(window)
        .corr(grupo['juros_real'])
    )

    # atribui os resultados ao dataframe original
    df_corr.loc[grupo.index, 'corr_IPCA_SELIC_acum_12m'] = corr_ipca_selic
    df_corr.loc[grupo.index, 'corr_IPCA_juros_real_12m'] = corr_ipca_jr

### Z-Score (Detecção de Anomalias)

O Z-Score mede quantos desvios-padrão uma observação está
acima ou abaixo da média.

Fórmula:

$$
Z = \frac{x - \mu}{\sigma}
$$

Onde:

- x  = valor do indicador (ex: IPCA_acum_12m, SELIC_12m, juros_real)  
- μ  = média do indicador no período (ou no cenário)
- σ  = desvio padrão do indicador no período (ou no cenário)

Interpretação:

- |Z| > 2 → possível anomalia  
- |Z| > 3 → forte evidência de outlier  
- Z ≈ 0   → valor próximo da média  

In [99]:
# Parâmetro da janela temporal
# Define a janela da correlação rolante (em meses)
window = 12  # correlação móvel de 12 meses

In [100]:
# DataFrame de trabalho para cálculo de correlação
# Cria uma cópia do dataframe original para evitar alterar
# o dataset principal durante os cálculos
df_z  = df_corr.copy()

In [101]:
# Cálculo de Z-Score rolante para indicadores principais
# Indicadores principais
colunas = ['IPCA_acum_12m','SELIC_acum_12m','juros_real']

# Loop por cenário
for cenario, grupo in df_z.groupby('cenarios'):

    # Ordena o grupo por data (boa prática)
    grupo = grupo.sort_values('data')

    for col in colunas:

        # Média e desvio padrão rolante da série
        rol_media = grupo[col].rolling(window).mean()
        rol_std = grupo[col].rolling(window).std()

        # Z-Score rolante: mede a distância em desvios-padrão
        z = (grupo[col] - rol_media) / rol_std

        # Atribui resultado ao dataframe original
        df_z.loc[grupo.index, f'Z_{col}_12m'] = z

### IQR (Outliers Estruturais)

O IQR (Interquartile Range) é uma medida de dispersão baseada
nos quartis da distribuição e é amplamente utilizado para
identificar outliers estruturais.

Definição:

$$IQR = Q3 - Q1$$

Onde:

- Q1 = primeiro quartil (25%)
- Q3 = terceiro quartil (75%)
- IQR = intervalo interquartil

Critério para detecção de outliers:

- Limite inferior = Q1 - 1.5 × IQR
- Limite superior = Q3 + 1.5 × IQR

Interpretação:

- Valores abaixo do limite inferior → possível outlier inferior
- Valores acima do limite superior → possível outlier superior
  
True → o valor está fora da faixa esperada (Q1 – 1.5IQR ou Q3 + 1.5IQR) → possível anomalia

False → o valor está dentro da faixa normal → considerado comportamento esperado

Observação:

O método do IQR é robusto a valores extremos,
pois não depende da média nem do desvio padrão.

In [103]:
# Parâmetro da janela temporal
# Define a janela da correlação rolante (em meses)
window = 12  # correlação móvel de 12 meses

In [104]:
# DataFrame de trabalho para cálculo de correlação
# Cria uma cópia do dataframe original para evitar alterar
# o dataset principal durante os cálculos
df_iqr = df_z.copy()

In [105]:
# Cálculo de IQR e detecção de outliers rolante
colunas = ['IPCA_acum_12m','SELIC_acum_12m','juros_real']

# Colunas de interesse
colunas = ['IPCA_acum_12m','SELIC_acum_12m','juros_real']

# Loop por cenário
for cenario, grupo in df_iqr.groupby('cenarios'):

    # Ordena o grupo por data (boa prática)
    grupo = grupo.sort_values('data')

    for col in colunas:

        # Calcula Q1 e Q3 rolante (percentis 25% e 75%)
        Q1 = grupo[col].rolling(window).quantile(0.25)
        Q3 = grupo[col].rolling(window).quantile(0.75)

        # Intervalo interquartil
        IQR = Q3 - Q1

        # Limites para detecção de outliers
        limite_inf = Q1 - 1.5 * IQR
        limite_sup = Q3 + 1.5 * IQR

        # Identificação de outliers (True se estiver fora dos limites)
        outlier = (grupo[col] < limite_inf) | (grupo[col] > limite_sup)

        # Armazena resultados no dataframe original
        df_iqr.loc[grupo.index, f'IQR_{col}_12m'] = IQR
        df_iqr.loc[grupo.index, f'outlier_{col}_12m'] = outlier